# Relacion entre materias primas y precio de los equipos

Parto de `dataset_equipos_limpio.csv` (salida de `01_limpieza_datos.ipynb`) para ver que tanto explica cada materia prima (X, Y, Z) al precio de cada equipo, con que rezago, y si es una relacion real o solo dos series con tendencia que suben juntas (correlacion espuria).

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

from statsmodels.tsa.stattools import adfuller, kpss, coint, grangercausalitytests
from statsmodels.tsa.vector_ar.vecm import coint_johansen
from statsmodels.tsa.api import VAR
from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.linear_model import LassoCV, ElasticNetCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, cross_val_score

import shap

PROCESSED_DIR = Path("processed")
REPORTS_DIR = Path("../reports")
REPORTS_DIR.mkdir(exist_ok=True)

df = pd.read_csv(PROCESSED_DIR / "dataset_equipos_limpio.csv", parse_dates=["date"]).set_index("date")

cols = ["price_x", "price_y", "price_z", "price_equipo1", "price_equipo2"]
materias = ["price_x", "price_y", "price_z"]
equipos = ["price_equipo1", "price_equipo2"]
nombres = {"price_x": "X", "price_y": "Y", "price_z": "Z", "price_equipo1": "Equipo1", "price_equipo2": "Equipo2"}

df.shape, df.index.min(), df.index.max()

Paleta y estilo fijos para todos los graficos del notebook: un color por serie, y una escala divergente azul-gris-rojo para correlaciones (gris = sin correlacion).

In [ ]:
PALETTE = {
    "price_x": "#2a78d6",        # azul
    "price_y": "#eb6834",        # naranja
    "price_z": "#1baf7a",        # aqua
    "price_equipo1": "#eda100",  # amarillo
    "price_equipo2": "#e87ba4",  # magenta
}
INK, INK_SEC, INK_MUT, GRID = "#0b0b0b", "#52514e", "#898781", "#e1e0d9"
DIVERGING_CMAP = LinearSegmentedColormap.from_list("diverging", ["#2a78d6", "#f0efec", "#e34948"])

plt.rcParams.update({
    "axes.edgecolor": GRID, "axes.labelcolor": INK_SEC, "xtick.color": INK_MUT, "ytick.color": INK_MUT,
    "text.color": INK, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
})

## Paso 0 — Granularidad

¿`X`, `Y`, `Z` cotizan a diario o son precios "pegados" (listas que se actualizan cada cierto tiempo)? Mido que % de dias repite el valor del dia anterior.

In [ ]:
rows_gran = []
for c in cols:
    igual = (df[c] == df[c].shift(1)).mean()
    cambia = df[c] != df[c].shift(1)
    racha = df.groupby(cambia.cumsum()).size()
    rows_gran.append({
        "serie": nombres[c],
        "repite_dia_anterior": igual,
        "racha_mediana_dias": racha.median(),
        "racha_maxima_dias": racha.max(),
    })

tabla_granularidad = pd.DataFrame(rows_gran).set_index("serie")
tabla_granularidad.style.format({"repite_dia_anterior": "{:.1%}"})

`Y` repite el valor del dia anterior en **55.5%** de las observaciones (rachas de hasta 35 dias): es un precio "pegado" a una lista, no una cotizacion diaria. `X` (0.3%) y `Z` (4.1%) si son casi-diarias; los dos equipos tambien. Por eso el CCF y Granger (Paso 4) trabajan sobre retornos, no niveles: en niveles estos tramos planos inflarian la autocorrelacion de forma artificial.

In [ ]:
idx = df[cols] / df[cols].iloc[0] * 100

fig, ax = plt.subplots(figsize=(10, 5))
# equipos primero (punteado), materias primas encima (solido): Y y Equipo1 casi se superponen
# (corr. 0.997 en niveles, Paso 2) y sin este orden la linea de Y queda tapada
for c in equipos:
    ax.plot(idx.index, idx[c], color=PALETTE[c], linewidth=1.8, linestyle=(0, (4, 2)), alpha=0.9, label=nombres[c])
for c in materias:
    ax.plot(idx.index, idx[c], color=PALETTE[c], linewidth=1.3, label=nombres[c])

ax.set_title("Series normalizadas (base 100 = 2010-01-04)", loc="left", fontsize=12, color=INK)
ax.set_ylabel("Indice (base 100)")
ax.legend(frameon=False, ncols=5, loc="upper left", fontsize=9)
fig.tight_layout()
plt.show()

## Paso 1 — Estacionariedad

ADF (H0: raiz unitaria) + KPSS (H0: estacionaria) sobre log-precio y sobre retornos (`diff` del log-precio) de las 5 series. Niveles con constante+tendencia (`ct`), retornos con constante (`c`).

In [ ]:
log_df = np.log(df[cols])
ret_df = log_df.diff().dropna()

def test_estacionariedad(serie, adf_reg, kpss_reg):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        adf_stat, adf_p, *_ = adfuller(serie, regression=adf_reg, autolag="AIC")
        kpss_stat, kpss_p, *_ = kpss(serie, regression=kpss_reg, nlags="auto")
    return adf_stat, adf_p, kpss_stat, kpss_p

def concluir(adf_p, kpss_p):
    adf_estac, kpss_estac = adf_p < 0.05, kpss_p > 0.05
    if adf_estac and kpss_estac:
        return "estacionaria"
    if not adf_estac and not kpss_estac:
        return "no estacionaria"
    return "ambigua"

filas = []
for c in cols:
    a_stat, a_p, k_stat, k_p = test_estacionariedad(log_df[c], "ct", "ct")
    filas.append({"serie": nombres[c], "tipo": "nivel (log)", "adf_stat": a_stat, "adf_p": a_p,
                  "kpss_stat": k_stat, "kpss_p": k_p, "conclusion": concluir(a_p, k_p)})
    a_stat, a_p, k_stat, k_p = test_estacionariedad(ret_df[c], "c", "c")
    filas.append({"serie": nombres[c], "tipo": "retorno (dlog)", "adf_stat": a_stat, "adf_p": a_p,
                  "kpss_stat": k_stat, "kpss_p": k_p, "conclusion": concluir(a_p, k_p)})

tabla_estacionariedad = pd.DataFrame(filas)
tabla_estacionariedad

Las 5 series son **I(1) en niveles** (ADF no rechaza raiz unitaria, `p` 0.48-0.68; KPSS rechaza estacionariedad, `p = 0.01`) y **I(0) en retornos** (ADF `p` ≈ 0; KPSS `p = 0.10`). Confirma por que correlacionar precios crudos es enganoso: dos series I(1) casi siempre salen correlacionadas en niveles sin relacion causal real ("spurious regression", Granger & Newbold 1974). En retornos si es interpretable.

## Paso 2 — Correlacion: niveles vs. retornos

Pearson y Spearman, en niveles y en retornos, lado a lado.

In [ ]:
pearson_niveles = df[cols].corr(method="pearson")
pearson_retornos = ret_df.corr(method="pearson")
spearman_niveles = df[cols].corr(method="spearman")
spearman_retornos = ret_df.corr(method="spearman")

print("Pearson, niveles:")
display(pearson_niveles.round(2))
print("Pearson, retornos:")
display(pearson_retornos.round(2))
print("Spearman, niveles:")
display(spearman_niveles.round(2))
print("Spearman, retornos:")
display(spearman_retornos.round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
for ax, mat, titulo in zip(axes, [pearson_niveles, pearson_retornos], ["Niveles (precio)", "Retornos (dlog)"]):
    im = ax.imshow(mat, cmap=DIVERGING_CMAP, vmin=-1, vmax=1)
    labels = [nombres[c] for c in mat.columns]
    ax.set_xticks(range(len(labels)), labels, fontsize=9)
    ax.set_yticks(range(len(labels)), labels, fontsize=9)
    for i in range(len(labels)):
        for j in range(len(labels)):
            v = mat.values[i, j]
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8,
                     color="#ffffff" if abs(v) > 0.6 else INK)
    ax.set_title(titulo, loc="left", fontsize=11, color=INK)
    ax.grid(False)
fig.colorbar(im, ax=axes, shrink=0.8, label="correlacion de Pearson")
fig.suptitle("Correlacion: niveles vs. retornos", x=0.02, ha="left", fontsize=12, color=INK)
plt.show()

Contraste enorme. En niveles, `Y`-`Equipo1` correlaciona **0.997** y `Z`-`Equipo2` **0.98**; en retornos caen a **0.38** y **0.40** — siguen siendo las relaciones mas fuertes del sistema, pero ya no parecen "la misma serie". `X` pasa de 0.48-0.53 en niveles a 0.02-0.07 en retornos: su correlacion en niveles era casi toda espuria. Spearman muestra el mismo patron.

## Paso 3 — Cointegracion

¿Hay equilibrio de largo plazo entre niveles, o solo tendencias parecidas por casualidad? Engle-Granger bivariado (cada materia vs. cada equipo) + Johansen multivariado (sistema `X, Y, Z, Equipo_i`).

In [ ]:
filas_eg = []
for m in materias:
    for e in equipos:
        stat, pval, _ = coint(log_df[e], log_df[m])
        filas_eg.append({"materia": nombres[m], "equipo": nombres[e], "eg_stat": stat, "eg_p": pval,
                          "cointegra_5pct": pval < 0.05})
tabla_engle_granger = pd.DataFrame(filas_eg)
tabla_engle_granger

`Y` y `Z` cointegran con ambos equipos al 5% (`p` 0.002-0.022); `X` no cointegra con ninguno (`p = 0.46` y `0.39`). Para el Johansen, elijo el rezago con seleccion de orden estandar sobre el VAR en niveles en vez de fijarlo a ojo.

In [ ]:
for e in equipos:
    print(f"--- seleccion de orden VAR, sistema (X,Y,Z,{nombres[e]}) ---")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        sel = VAR(log_df[["price_x", "price_y", "price_z", e]]).select_order(maxlags=10)
    print(sel.summary())

BIC y HQIC coinciden en orden bajo (1-2); uso el mas parsimonioso que valida HQIC: **orden 2 => `k_ar_diff = 1`**.

In [ ]:
K_AR_DIFF = 1
filas_johansen = []
vectores_johansen = {}

for e in equipos:
    sub = log_df[["price_x", "price_y", "price_z", e]]
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")  # advertencia benigna: la parte imaginaria de los autovalores es ~0
        res = coint_johansen(sub, det_order=0, k_ar_diff=K_AR_DIFF)

    trace_stat, trace_crit5 = res.lr1, res.cvt[:, 1]
    maxeig_stat, maxeig_crit5 = res.lr2, res.cvm[:, 1]
    rango_trace = int((trace_stat > trace_crit5).sum())
    rango_maxeig = int((maxeig_stat > maxeig_crit5).sum())

    filas_johansen.append({"equipo": nombres[e], "rango_trace_5pct": rango_trace,
                            "rango_maxeig_5pct": rango_maxeig})

    evec1 = res.evec[:, 0].real
    evec1 = evec1 / evec1[0]
    vectores_johansen[nombres[e]] = pd.Series(evec1, index=["X", "Y", "Z", nombres[e]]).round(3)

    print(f"--- sistema (X,Y,Z,{nombres[e]}) ---")
    print("trace stat:", np.round(trace_stat, 1), " crit 5%:", np.round(trace_crit5, 1))
    print("max-eig stat:", np.round(maxeig_stat, 1), " crit 5%:", np.round(maxeig_crit5, 1))
    print()

tabla_johansen = pd.DataFrame(filas_johansen)
tabla_johansen

Trace y max-eigenvalue sugieren rango 4 (cointegracion completa), lo cual contradice el Paso 1 (I(1) individual) y a Engle-Granger (`X` no cointegra). No lo tomo al pie de la letra: Johansen asume residuos homocedasticos, y series financieras diarias con *volatility clustering* + muestra grande (~3500 obs) tipicamente inflan el estadistico y sobre-rechazan la hipotesis nula. Lo uso solo como senal secundaria, via el vector de cointegracion:

In [ ]:
for equipo_nombre, vec in vectores_johansen.items():
    print(f"{equipo_nombre}: {vec.to_dict()}")

Normalizando `X=1`: vector de `Equipo1` = `Y=29.5, Z=-0.16, Equipo1=-30.4` (domina `Y`); vector de `Equipo2` = `Y=7.2, Z=26.9, Equipo2=-35.4` (domina `Z`). Coincide con Engle-Granger: Y para Equipo1, Z para Equipo2, X marginal en ambos — mismo patron que aparece otra vez en las secciones 4 y 5.

## Paso 4 — CCF y causalidad de Granger

Sobre retornos (estacionarios): correlacion entre `retorno_materia(t-k)` y `retorno_equipo(t)`, `k` en ±20 dias habiles (`k>0`: la materia prima lidera).

In [ ]:
MAX_LAG = 20

def ccf_lags(x, y, max_lag):
    out = {}
    for k in range(-max_lag, max_lag + 1):
        if k >= 0:
            a, b = x.shift(k).iloc[max_lag:], y.iloc[max_lag:]
        else:
            a, b = x.iloc[max_lag:], y.shift(-k).iloc[max_lag:]
        out[k] = a.corr(b)
    return pd.Series(out)

ccf_por_par = {}
filas_ccf = []
for m in materias:
    for e in equipos:
        s = ccf_lags(ret_df[m], ret_df[e], MAX_LAG)
        ccf_por_par[(m, e)] = s
        mejor_lag = s.abs().idxmax()
        filas_ccf.append({"materia": nombres[m], "equipo": nombres[e], "lag_optimo": mejor_lag,
                           "corr_en_lag_optimo": s[mejor_lag]})

tabla_ccf = pd.DataFrame(filas_ccf)
tabla_ccf

In [ ]:
fig, axes = plt.subplots(len(materias), len(equipos), figsize=(9, 8), sharex=True)
for i, m in enumerate(materias):
    for j, e in enumerate(equipos):
        ax = axes[i, j]
        s = ccf_por_par[(m, e)]
        colores = [PALETTE[m] if k >= 0 else INK_MUT for k in s.index]
        ax.bar(s.index, s.values, color=colores, width=0.8)
        ax.axhline(0, color=GRID, linewidth=0.8)
        ax.set_title(f"{nombres[m]} -> {nombres[e]}", fontsize=9, loc="left", color=INK)
        if i == len(materias) - 1:
            ax.set_xlabel("rezago (dias)", fontsize=8)
fig.suptitle("Correlacion cruzada (CCF) sobre retornos, rezago +-20 dias habiles", x=0.02, ha="left", fontsize=12, color=INK)
fig.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

Pico dominante en **rezago 0** para `Y -> Equipo1` (0.38) y `Z -> Equipo2` (0.40): relacion contemporanea, no lead-lag. `X` no muestra ningun pico claro en ninguno de los dos equipos. Formalizo con Granger (hasta 5 rezagos):

In [ ]:
filas_granger = []
for m in materias:
    for e in equipos:
        datos = ret_df[[e, m]].dropna()  # orden [y, x]: testea si "m" Granger-causa "e"
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            res = grangercausalitytests(datos, maxlag=5)
        pvals = {lag: res[lag][0]["ssr_ftest"][1] for lag in res}
        mejor_lag = min(pvals, key=pvals.get)
        filas_granger.append({"materia": nombres[m], "equipo": nombres[e], "mejor_lag": mejor_lag,
                               "p_valor": pvals[mejor_lag], "granger_causa_5pct": pvals[mejor_lag] < 0.05})

tabla_granger = pd.DataFrame(filas_granger)
tabla_granger

Las 6 combinaciones salen significativas al 5% — con ~3500 obs el test tiene mucha potencia, asi que hasta un efecto chico (como `X`) da `p` bajo. No uso esta tabla sola: la leo junto con la magnitud del CCF y los coeficientes del Paso 5.

## Paso 5 — Seleccion de variables y cuantificacion

Retornos de `X`, `Y`, `Z` en rezagos 0-5 dias. VIF antes de interpretar coeficientes, Lasso/Elastic Net con `TimeSeriesSplit`, y Random Forest + SHAP como contraste no lineal.

In [ ]:
X_ret_c = ret_df[materias].assign(const=1.0)
vif = pd.DataFrame({
    "materia": [nombres[m] for m in materias],
    "VIF": [variance_inflation_factor(X_ret_c.values, i) for i in range(len(materias))],
})
vif

VIF ≈ 1 para las 3 (1.01-1.04): sin multicolinealidad relevante entre `X`, `Y`, `Z` en retornos.

In [ ]:
def construir_features(ret_df, materias, equipo, n_lags=5):
    feats = {f"{nombres[m]}_lag{lag}": ret_df[m].shift(lag) for m in materias for lag in range(n_lags + 1)}
    X = pd.DataFrame(feats, index=ret_df.index)
    y = ret_df[equipo]
    ambos = pd.concat([X, y.rename("target")], axis=1).dropna()
    return ambos.drop(columns="target"), ambos["target"]

tscv = TimeSeriesSplit(n_splits=5)
resultados_lineales = {}

for e in equipos:
    X, y = construir_features(ret_df, materias, e)

    lasso = LassoCV(cv=tscv, random_state=42, max_iter=20000).fit(X, y)
    enet = ElasticNetCV(cv=tscv, random_state=42, max_iter=20000, l1_ratio=[.1, .5, .7, .9, .95, 1]).fit(X, y)

    coef_lasso = pd.Series(lasso.coef_, index=X.columns)
    coef_enet = pd.Series(enet.coef_, index=X.columns)
    resultados_lineales[e] = {"X": X, "y": y, "lasso": lasso, "enet": enet,
                               "coef_lasso": coef_lasso, "coef_enet": coef_enet}

    print(f"=== {nombres[e]} ===")
    print(f"Lasso:        alpha={lasso.alpha_:.6f}  R2 in-sample={lasso.score(X, y):.3f}  "
          f"variables no nulas={int((coef_lasso != 0).sum())}/{len(coef_lasso)}")
    print(f"Elastic Net:  alpha={enet.alpha_:.6f}  l1_ratio={enet.l1_ratio_:.2f}  "
          f"R2 in-sample={enet.score(X, y):.3f}  variables no nulas={int((coef_enet != 0).sum())}/{len(coef_enet)}")
    print("Top coeficientes Lasso (por magnitud):")
    print(coef_lasso.reindex(coef_lasso.abs().sort_values(ascending=False).index).head(6).to_string())
    print()

Lasso y Elastic Net coinciden: un solo rezago concentra casi todo el coeficiente — `Y_lag0` para Equipo1, `Z_lag0` para Equipo2 —, el resto en ~0. R² in-sample modesto (0.14-0.17): estos insumos explican una parte real pero parcial (quedan fuera mano de obra, logistica, margen). Contrasto con Random Forest + SHAP.

In [ ]:
resultados_rf = {}
for e in equipos:
    X, y = resultados_lineales[e]["X"], resultados_lineales[e]["y"]
    rf = RandomForestRegressor(n_estimators=300, max_depth=5, min_samples_leaf=20, random_state=42, n_jobs=-1)
    r2_cv = cross_val_score(rf, X, y, cv=tscv, scoring="r2")
    rf.fit(X, y)

    explainer = shap.TreeExplainer(rf)
    shap_values = explainer.shap_values(X)
    shap_abs_medio = pd.Series(np.abs(shap_values).mean(axis=0), index=X.columns)
    shap_por_materia = pd.Series({
        nombres[m]: shap_abs_medio[[c for c in X.columns if c.startswith(nombres[m] + "_lag")]].sum()
        for m in materias
    }).sort_values(ascending=False)

    resultados_rf[e] = {"rf": rf, "r2_cv": r2_cv, "shap_values": shap_values, "shap_por_materia": shap_por_materia}

    print(f"=== {nombres[e]} ===")
    print(f"R2 (TimeSeriesSplit, 5 folds): {np.round(r2_cv, 3)}  media={r2_cv.mean():.3f}")
    print("Importancia SHAP agregada por materia prima (suma |SHAP| de todos los rezagos):")
    print(shap_por_materia.round(5).to_string())
    print()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
color_por_materia = {"X": PALETTE["price_x"], "Y": PALETTE["price_y"], "Z": PALETTE["price_z"]}
for ax, e in zip(axes, equipos):
    agg = resultados_rf[e]["shap_por_materia"].sort_values()
    ax.barh(agg.index, agg.values, color=[color_por_materia[k] for k in agg.index], height=0.55)
    ax.set_title(nombres[e], loc="left", fontsize=11, color=INK)
    ax.set_xlabel("|SHAP| medio (suma de rezagos 0-5)", fontsize=9)
fig.suptitle("Importancia SHAP agregada por materia prima (Random Forest)", x=0.02, ha="left", fontsize=12, color=INK)
fig.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()

Mismo resultado por un camino no lineal: `Y` domina Equipo1, `Z` domina Equipo2, `X` aporta poco en ambos. R² de validacion cruzada similar al de Lasso (~0.10-0.12): sin evidencia de relaciones no lineales relevantes que el modelo lineal se este perdiendo. 5 metodos independientes coinciden en la misma conclusion.

## Paso 6 — Tabla resumen por equipo

Signo y magnitud (coeficiente Lasso dominante), rezago optimo (CCF), cointegra (Engle-Granger) y Granger-causa, por combinacion materia prima-equipo.

In [ ]:
filas_resumen = []
for m in materias:
    for e in equipos:
        coef_lasso = resultados_lineales[e]["coef_lasso"]
        cols_materia = [c for c in coef_lasso.index if c.startswith(nombres[m] + "_lag")]
        coef_dominante = coef_lasso[cols_materia].reindex(coef_lasso[cols_materia].abs().sort_values(ascending=False).index).iloc[0]
        rezago_dominante = coef_lasso[cols_materia].abs().idxmax().split("_lag")[1]

        eg_row = tabla_engle_granger[(tabla_engle_granger["materia"] == nombres[m]) & (tabla_engle_granger["equipo"] == nombres[e])].iloc[0]
        gr_row = tabla_granger[(tabla_granger["materia"] == nombres[m]) & (tabla_granger["equipo"] == nombres[e])].iloc[0]
        ccf_row = tabla_ccf[(tabla_ccf["materia"] == nombres[m]) & (tabla_ccf["equipo"] == nombres[e])].iloc[0]
        shap_val = resultados_rf[e]["shap_por_materia"][nombres[m]]

        filas_resumen.append({
            "equipo": nombres[e],
            "insumo": nombres[m],
            "signo_lasso": "+" if coef_dominante > 0 else ("-" if coef_dominante < 0 else "0"),
            "coef_lasso_dominante": coef_dominante,
            "rezago_lasso_dominante": rezago_dominante,
            "lag_optimo_ccf": ccf_row["lag_optimo"],
            "corr_ccf": ccf_row["corr_en_lag_optimo"],
            "cointegra_eg_5pct": eg_row["cointegra_5pct"],
            "granger_causa_5pct": gr_row["granger_causa_5pct"],
            "shap_medio": shap_val,
        })

tabla_resumen = pd.DataFrame(filas_resumen).sort_values(["equipo", "shap_medio"], ascending=[True, False]).reset_index(drop=True)
tabla_resumen

In [ ]:
tabla_resumen.to_csv(PROCESSED_DIR / "resumen_relacion_materias_primas_equipos.csv", index=False)
print("Guardado en data/processed/resumen_relacion_materias_primas_equipos.csv:", tabla_resumen.shape)

## Conclusiones

- **Equipo1 ≈ `Y`** (cointegra, domina Johansen, CCF, Lasso/Elastic Net y SHAP). Como `Y` es un precio   "pegado" (Paso 0), se lee mejor como indexacion a una lista de precios que como transmision de mercado.
- **Equipo2 ≈ `Z`**, mismo patron de evidencia en las 5 secciones.
- **`X` es marginal en ambos**: no cointegra, correlacion en retornos casi nula (0.03-0.07), menor   importancia en Lasso y SHAP. Su correlacion alta en niveles era casi toda espuria.
- La relacion es siempre **contemporanea** (rezago 0), no anticipable con dias de anticipacion.
- El Johansen sugirio rango completo, pero no lo tomo al pie de la letra (ver Paso 3); lo uso solo como   evidencia secundaria.
- R² modestos en todos los modelos (0.10-0.17): estos insumos explican una parte real pero parcial del   precio, quedan fuera otros costos (mano de obra, logistica, margen).

Tabla resumen en `data/processed/resumen_relacion_materias_primas_equipos.csv`, insumo para la Fase 3 (forecasting): `Y` para Equipo1, `Z` para Equipo2, rezago 0.